# Building Tools for Agents

*Notebook 02*

Tools let an agent do work outside the model.

In this lesson, a Python function becomes a callable tool with `@function_tool`.

---

## 🔧 Setup

In [ ]:
from datetime import datetime
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(dotenv_path=Path("..") / ".env")

from agents import Agent, Runner, function_tool

MODEL = "gpt-5-mini"  # Course default, fast and cost-effective for most tasks

print("✅ Ready!")

---

## 🛠️ Part 1: Your First Tool

Without tools, the agent can only answer from the model.

It cannot query your private data or live systems.

A tool gives it a Python function to call.

### Before: Agent without tools

In [ ]:
message = "What department does employee E003 work in?"

agent_no_tools = Agent(
    name="NoToolsAgent",
    instructions="You are a helpful assistant.",
    model=MODEL
)

result = await Runner.run(agent_no_tools, input=message)

print(result.final_output)

### After: Agent with an employee lookup tool

In [ ]:
# Simulated employee database
EMPLOYEES = {
    "E001": {"name": "Sarah Chen", "department": "Engineering", "salary": 95000},
    "E002": {"name": "Marcus Johnson", "department": "Marketing", "salary": 78000},
    "E003": {"name": "Priya Patel", "department": "Finance", "salary": 88000},
    "E004": {"name": "Tom Rivera", "department": "Engineering", "salary": 102000},
    "E005": {"name": "Lisa Wong", "department": "HR", "salary": 72000},
}

@function_tool
def get_employee(employee_id: str) -> str:
    """Look up an employee's name, department, and salary by their employee ID."""
    employee = EMPLOYEES.get(employee_id.upper())
    if employee:
        return (
            f"Name: {employee['name']}, "
            f"Department: {employee['department']}, "
            f"Salary: ${employee['salary']:,}"
        )
    return f"No employee found with ID {employee_id}"

# --------------------------------------------------------------
print("✅ get_employee() ready")

The `@function_tool` decorator exposes three signals:

- **Function name:** what the tool is called

- **Docstring:** what the tool does and when to use it

- **Type hints:** the input schema, no manual JSON

#### Run Agent with Employee Lookup Tool

In [ ]:
agent_with_tools = Agent(
    name="EmployeeAgent",
    instructions="You are a helpful assistant.",
    model=MODEL,
    tools=[get_employee]
)

result = await Runner.run(agent_with_tools, input=message)

print(result.final_output)

Open Traces and check the middle step.

The agent called `get_employee` with the employee ID.

---

## 🗓️ Part 2: A Tool with No Parameters

Not all tools take input.

Some just return live information: the current time, system status, environment data.

In [ ]:
@function_tool
def get_current_datetime() -> str:
    """Return the current date and time."""
    return datetime.now().strftime("%A, %B %d, %Y at %I:%M %p")

# --------------------------------------------------------------
print("✅ get_current_datetime() ready")

#### Run Agent with DateTime Tool

In [ ]:
agent_datetime = Agent(
    name="DateTimeAgent",
    instructions="Use the get_current_datetime tool for all time and date questions.",
    model=MODEL,
    tools=[get_current_datetime]
)

result = await Runner.run(agent_datetime, input="What day of the week is it today?")

print(result.final_output)

---

## 🧠 Part 3: How the Agent Decides

The agent does not call every tool.

It chooses from the user's message, the tool name, and the docstring.

That means your docstring is part of the prompt.

Clear docstrings lead to better tool choices.

In [ ]:
# Agent with both tools, so we can watch it choose.
agent_multi = Agent(
    name="MultiToolAgent",
    instructions="You are a helpful assistant. Use tools whenever they are relevant.",
    model=MODEL,
    tools=[get_employee, get_current_datetime]
)

# --------------------------------------------------------------
print("✅ MultiToolAgent ready")

#### Test: Employee Question

In [ ]:
result = await Runner.run(agent_multi, input="What is employee E001's salary?")

print(result.final_output)

#### Test: Time Question

In [ ]:
result = await Runner.run(agent_multi, input="What time is it right now?")

print(result.final_output)

#### Test: No Tool Needed

In [ ]:
result = await Runner.run(agent_multi, input="What is the capital of France?")

print(result.final_output)

### 💡 Writing a Good Tool Docstring

A useful docstring tells the agent:

1. What the tool returns

2. When to use it

3. What each input means

`"Get employee info"` is vague.

A better docstring names the fields and the use case.

---

## 💪 Practice Exercises

### Exercise 1: Word Counter Tool

*Covers: `@function_tool`: building and wiring custom tools*

Create a word-count tool, then build an agent that uses it.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 1: Word Counter Tool
# --------------------------------------------------------------
# Objective: Build a tool that counts words and an agent that uses it.

# TODO 1: Define a @function_tool called count_words
#             - Parameter: text (str)
#             - Returns: str with the word count
#             - Docstring: describe what it does clearly
#             - Hint: len(text.split()) gives you the word count

# TODO 2: Create an Agent with the count_words tool

# TODO 3: Run the agent with: "How many words are in this sentence: The quick brown fox"

# --- Write your code below this line ---

### Exercise 2: Temperature Converter

*Covers: `@function_tool`: tool inputs, outputs, and docstrings*

Create a Celsius-to-Fahrenheit tool, then build an agent that uses it.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 2: Temperature Converter
# --------------------------------------------------------------
# Objective: Build a tool that converts temperatures.

# TODO 1: Define a @function_tool called celsius_to_fahrenheit
#             - Parameter: celsius (float)
#             - Formula: (celsius * 9/5) + 32
#             - Returns: str with the result

# TODO 2: Create an Agent with the conversion tool

# TODO 3: Run the agent with: "What is 100 degrees Celsius in Fahrenheit?"

# --- Write your code below this line ---

---

## 🎯 Key Takeaways

**`@function_tool` turns Python functions into agent capabilities:**

- The function name becomes the tool name

- The docstring tells the agent when to use it

- Type hints generate the input schema, no manual JSON
<br>
<br>

**Tool selection is automatic:**

- The agent reads the user's message, tool names, and docstrings

- A clear docstring helps the agent choose the right tool

- No tool needed means no tool called

---

## 📍 Next Step

**Notebook 03: Writing Effective Agent Instructions**

Learn to write system instructions that shape behavior, tone, and tool use.

---

##### 🔧 [Troubleshooting Guide](https://github.com/barrettscott/openai-agents/blob/main/TROUBLESHOOTING.md#lesson-02-building-tools-for-agents)

---